In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit import als
import pickle
import os

# Load clean data
df = pd.read_csv(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\clean_ratings.csv')
print(df.shape)
print(df.head())

(72696, 6)
          user_id  product_id  rating  \
0  A3FO5AKVTFRCRJ  0739079891     5.0   
1  A1H4I7VO8YVMEI  0739079891     1.0   
2  A1NZF2Y9MQD5QA  0739079891     4.0   
3  A24K057TXUJASY  0739079891     5.0   
4  A1XN2ORFU4W0SY  0739079891     5.0   

                                         review_text  \
0                            It's good for beginners   
1  Great concept with great instructions and CDs....   
2                                good starter set up   
3  I am enjoying my ukulele. I had one a long tim...   
4                                               Good   

                     summary   timestamp  
0                 Five Stars  1477785600  
1  Frustratingly out of tune  1422403200  
2                 Four Stars  1420329600  
3       Ukulele starter pack  1419552000  
4                 Five Stars  1416355200  


In [2]:
# ML models need numbers, not string IDs
# We create a mapping: original_id <-> integer index

user_encoder = {uid: idx for idx, uid in enumerate(df['user_id'].unique())}
product_encoder = {pid: idx for idx, pid in enumerate(df['product_id'].unique())}

# Reverse mappings (we'll need these at inference time)
user_decoder = {idx: uid for uid, idx in user_encoder.items()}
product_decoder = {idx: pid for pid, idx in product_encoder.items()}

df['user_idx'] = df['user_id'].map(user_encoder)
df['product_idx'] = df['product_id'].map(product_encoder)

n_users = df['user_idx'].nunique()
n_products = df['product_idx'].nunique()

print(f"Users: {n_users}, Products: {n_products}")

Users: 10327, Products: 2717


In [3]:
# This is the core data structure of collaborative filtering
# Rows = users, Columns = products, Values = ratings
# It's a SPARSE matrix because most users haven't rated most products

# Convert rating to "confidence" — ALS works better with this
# Higher rating = higher confidence the user likes it
df['confidence'] = df['rating'].apply(lambda x: 1 + 40 * x)

# Build sparse matrix (items x users — implicit library expects this format)
interaction_matrix = csr_matrix(
    (df['confidence'], (df['product_idx'], df['user_idx'])),
    shape=(n_products, n_users)
)

print("Matrix shape:", interaction_matrix.shape)
print("Total interactions:", interaction_matrix.nnz)
sparsity = 1 - (interaction_matrix.nnz / (n_products * n_users))
print(f"Sparsity: {sparsity:.4%}")
# Expect ~99% sparsity — totally normal!

Matrix shape: (2717, 10327)
Total interactions: 64695
Sparsity: 99.7694%


In [4]:
# ALS = Alternating Least Squares
# It learns a 50-dimensional "taste vector" for each user
# and a 50-dimensional "feature vector" for each product
# Users and products with similar vectors = good match

model = als.AlternatingLeastSquares(
    factors=50,          # size of embedding vectors
    regularization=0.1,  # prevent overfitting
    iterations=30,       # training epochs
    use_gpu=False        # set True if you have a GPU
)

print("Training ALS model...")
model.fit(interaction_matrix)
print("✅ Training complete!")

Training ALS model...


c:\Users\bvais\anaconda3\envs\ml\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

✅ Training complete!


In [5]:
def recommend_for_user(user_id, n=10):
    if user_id not in user_encoder:
        print(f"User {user_id} not found")
        return []

    user_idx = user_encoder[user_id]

    # FIX: extract user row as a proper CSR matrix (not just a row slice)
    user_items = interaction_matrix.T.tocsr()[user_idx]

    recommended_ids, scores = model.recommend(
        user_idx,
        user_items,
        N=n,
        filter_already_liked_items=True
    )

    results = []
    for product_idx, score in zip(recommended_ids, scores):
        results.append({
            'product_id': product_decoder[product_idx],
            'score': round(float(score), 4)
        })

    return results

# Test it
sample_user = df['user_id'].iloc[0]
print(f"Recommendations for user: {sample_user}\n")
recs = recommend_for_user(sample_user)
for i, r in enumerate(recs, 1):
    print(f"{i}. Product: {r['product_id']}  Score: {r['score']}")

Recommendations for user: A3FO5AKVTFRCRJ

1. Product: B00001W0DH  Score: 0.9339
2. Product: B0002F75LA  Score: 0.9265
3. Product: B00004Y2V2  Score: 0.8914
4. Product: B000068NYM  Score: 0.8892
5. Product: B00000J50W  Score: 0.8861
6. Product: B0002F59ZY  Score: 0.8804
7. Product: B000068NSX  Score: 0.8757
8. Product: B000068NVJ  Score: 0.8699
9. Product: B00001W0DT  Score: 0.8662
10. Product: B00004UE29  Score: 0.8463


In [6]:
os.makedirs('../models', exist_ok=True)

# Save everything we need for inference later
with open('../models/als_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({
        'user_encoder': user_encoder,
        'user_decoder': user_decoder,
        'product_encoder': product_encoder,
        'product_decoder': product_decoder
    }, f)

with open('../models/interaction_matrix.pkl', 'wb') as f:
    pickle.dump(interaction_matrix, f)

print("✅ Model and encoders saved to models/")

✅ Model and encoders saved to models/
